In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

In [20]:
api_token = os.getenv("HUGGINGFACE_API_KEY")

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational",
    max_new_tokens=100,
    temperature=0.1,
    huggingfacehub_api_token=api_token,
)

chat_model = ChatHuggingFace(llm=llm)

In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [22]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [23]:
def generate_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f"Generate a detailed outline for a blog post with the title: {title}"
    outline = chat_model.invoke(prompt).content
    state['outline'] = outline
    return state

In [24]:
def generate_content(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f"Generate a detailed blog post based on the following title and outline:\nTitle: {title}\nOutline: {outline}"
    content = chat_model.invoke(prompt).content
    state['content'] = content
    return state

In [25]:
graph = StateGraph(BlogState)

#Add node
graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_content', generate_content)

#Add edges
graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_content')
graph.add_edge('generate_content', END)

#Compile
workflow = graph.compile()

In [26]:
initial_state = {'title': 'Manchester City Football Club'}

final_state = workflow.invoke(initial_state)

print("The outline generated is:")
print(final_state['outline'])

print("\n\nThe content generated is:")
print(final_state['content'])

The outline generated is:
Here's a detailed outline for a blog post on Manchester City Football Club:

**I. Introduction**

* Brief overview of Manchester City Football Club's history and significance
* Importance of the club in the English Premier League
* Thesis statement: Manchester City Football Club has evolved into one of the most successful and popular football clubs in the world, with a rich history, talented players, and a passionate fan base.

**II. History of Manchester City Football Club**

* Early years (1880-194


The content generated is:
**Manchester City Football Club: A Journey to the Top**

Manchester City Football Club, one of the most successful and popular football clubs in the world, has a rich history that spans over 140 years. From its humble beginnings in the late 19th century to its current status as a global football powerhouse, Manchester City has evolved into a club that is synonymous with excellence, passion, and dedication. In this blog post, we will del